# Cloud training: UTA-RLDD temporal drowsiness model

This Google Colab workflow uses the UTA Real-Life Drowsiness Dataset (UTA-RLDD), with a participant-level holdout and a temporal GRU. It trains an alert-versus-drowsy endpoint model; low-vigilance videos are intentionally excluded by the current project label mapping.

The official dataset is about 111 GB. Download it from the [UTA-RLDD page](https://sites.google.com/view/utarldd/home) to Google Drive first, preserving participant folders and video names `0`, `5`, and `10`. The dataset authors request citation of Ghoddoosian et al., CVPR Workshops 2019. Push the repository changes to GitHub first; the notebook clones the `main` branch. Select **Runtime → Change runtime type → T4 GPU** before running. Video decoding/landmark extraction is CPU work; the GRU training itself uses the GPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to the folder containing the participant directories.
DATASET_DIR = '/content/drive/MyDrive/UTA-RLDD'
OUTPUT_DIR = '/content/drive/MyDrive/SafeDrive-UTA-RLDD-output'


In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), 'Enable a GPU in Runtime settings before training.'
print(torch.cuda.get_device_name(0))
REPO = '/content/SafeDrive-Monitor'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Pravith101/SafeDrive-Monitor.git', REPO], check=True)
%cd /content/SafeDrive-Monitor
# Install only the libraries needed by video preprocessing and the temporal model.
%pip install -q opencv-contrib-python mediapipe scikit-learn joblib


In [ ]:
from pathlib import Path
dataset = Path(DATASET_DIR)
assert dataset.is_dir(), f'Dataset folder not found: {dataset}'
# Sample one frame per second to avoid running FaceMesh on every source frame.
subprocess.run([
    'python', 'data/preprocess_activity.py', '--dataset-dir', str(dataset),
    '--sample-fps', '1', '--sequence-length', '30', '--stride', '30'
], check=True)
subprocess.run(['python', 'models/temporal_gru.py'], check=True)


In [ ]:
# Preserve the model, its matching feature scaler, and split metadata together.
from pathlib import Path
import shutil
output = Path(OUTPUT_DIR)
output.mkdir(parents=True, exist_ok=True)
for src, name in [
    (Path('weights/temporal_gru.pth'), 'temporal_gru.pth'),
    (Path('data/processed/feature_scaler.joblib'), 'feature_scaler.joblib'),
    (Path('data/processed/metadata.json'), 'metadata.json'),
]:
    shutil.copy2(src, output / name)
print('Cloud artifacts saved to', output)


## Use the checkpoint locally

Copy `temporal_gru.pth` into the repository's `weights/` folder and copy `feature_scaler.joblib` into `data/processed/`. Then run `python live_inference.py --temporal`. Keep the checkpoint and scaler from the same training run together. The held-out score is participant-level validation, not a five-fold benchmark; the official dataset page recommends five-fold participant-level cross-validation for comparable research results. Do not treat this prototype as a safety-critical detector.
